# 03. Baseline Tokenizer Benchmark & Metrics Calibration
Evaluates representative multilingual tokenizers (mBERT, XLM-R, AfroXLMR) and Ghana-made ABENA against Twi, Ewe, and English text using token fertility and subword fragmentation metrics.

## 1. Initial Taste-Test on Isolated Greetings

In [1]:
import sys
sys.path.append("..")
from src.tokenizer_benchmark import load_tokenizers, load_twi_only_tokenizers, analyze_sentence

tokenizers = load_tokenizers()
twi_only_tokenizers = load_twi_only_tokenizers()

sample_twi = "Akwaaba, wo ho te sɛn?"
sample_ewe = "Woezɔ, aleke nèfɔ?"
sample_en  = "Welcome, how are you?"

print("=== MULTILINGUAL BASELINE TOKENIZERS ===")
for name, tok in tokenizers.items():
    print(f"\n{name} on Twi:     ", tok.tokenize(sample_twi))
    print(f"{name} on Ewe:     ", tok.tokenize(sample_ewe))
    print(f"{name} on English: ", tok.tokenize(sample_en))

print("\n=== GHANA-MADE TWI BASELINE (ABENA) ===")
for name, tok in twi_only_tokenizers.items():
    print(f"{name} on Twi: ", tok.tokenize(sample_twi))

### **Observations & Analysis: The African Language Tokenization Tax**

| Language | Sentence | Word Count | Tokens Produced (mBERT) | Tokens Produced (XLM-R) | Fertility (Tokens/Word) | Subword Fragmentation Rate |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **English** | *"Welcome, how are you?"* | 4 | **6** (`['Welcome', ',', 'how', 'are', 'you', '?']`) | **6** | **1.00** | **0.0%** (Whole-word preservation) |
| **Twi** | *"Akwaaba, wo ho te sɛn?"* | 4 | **11** (`['Ak', '##wa', '##aba', ',', 'wo', 'ho', 'te', 's', '##ɛ', '##n', '?']`) | **11** | **2.75** | **175% Overhead** (Splits on open vowels `ɛ`) |
| **Ewe** | *"Woezɔ, aleke nèfɔ?"* | 3 | **12** (`['Wo', '##ez', '##ɔ', ',', 'ale', '##ke', 'n', '##è', '##f', '##ɔ', '?']`) | **10** | **4.00** | **300% Overhead** (Severe character isolation) |

## 2. Core Tokenization Efficiency Metrics

* **`tokens_per_word` (Token Fertility)**: The average number of subword tokens produced per whitespace-separated word (calculated in isolation to avoid cross-boundary merge bias). A fertility score of 1.0 means whole-word tokenization. Lower is better. This is the primary headline metric.
* **`tokens_per_char`**: How many tokens are produced per character of text. Useful for evaluating representation density independent of language-specific average word length.
* **`pct_words_split` (Fragmentation Rate)**: The percentage of words split into two or more subword pieces. A high percentage indicates that the tokenizer vocabulary lacks whole-word representations for the target language.

In [2]:
# Sanity check on 5 processed sentences from the training corpus
with open("../data/processed/twi/train.txt", encoding="utf-8") as f:
    sample_sentences = [next(f).strip().strip('"') for _ in range(5)]

all_tokenizers = {**tokenizers, **twi_only_tokenizers}

for sentence in sample_sentences:
    print("\nSentence:", sentence)
    for name, tok in all_tokenizers.items():
        result = analyze_sentence(tok, sentence)
        print(f"  {name}: fertility={result['tokens_per_word']:.2f}, "
              f"pct_split={result['pct_words_split']:.1f}%, "
              f"tokens_per_char={result['tokens_per_char']:.3f}")